<a href="https://colab.research.google.com/github/arpitmehta0810/trap-trainer/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [3]:
   !pip install google-genai --quiet
   print("Done.")

Done.


In [4]:
DOCUMENTS = {
    "arrays": (
        "Arrays store elements in contiguous memory. "
        "Accessing an element by index takes O(1) time. "
        "Inserting or deleting in the middle takes O(n) because elements must shift. "
        "Arrays are cache friendly, which makes iterating over them fast."
    ),
    "linked_list": (
        "A linked list is made of nodes that each hold a value and a pointer to the next node. "
        "Inserting at the head takes O(1) time. "
        "Searching for a value takes O(n) because there is no random access. "
        "A doubly linked list also stores a pointer to the previous node, which makes deletion easier."
    ),
    "stack_queue": (
        "A stack follows last in first out order, and push and pop take O(1) time. "
        "A queue follows first in first out order, and enqueue and dequeue take O(1) time. "
        "Stacks are used for undo features and tracking recursive calls. "
        "Queues are used in breadth first search and task scheduling."
    ),
    "binary_search": (
        "Binary search finds a target in a sorted array by repeatedly halving the search range. "
        "It runs in O(log n) time. "
        "The array must be sorted before you start. "
        "Off by one errors in the loop bounds are the most common mistake."
    ),
}
print(f"Loaded {len(DOCUMENTS)} documents.")

Loaded 4 documents.


In [5]:
STOPWORDS = {"the", "a", "an", "is", "are", "of", "to", "in", "what", "how", "does", "do", "for", "and", "why", "which"}

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    words = {w.strip("?.,!").lower() for w in query.split()} - STOPWORDS
    results = []

    for doc_name, content in DOCUMENTS.items():
        content_lower = content.lower()
        match_count = sum(1 for word in words if word and word in content_lower)
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)

# Sanity check
print(search_documents("binary search"))

[Source: binary_search]
Binary search finds a target in a sorted array by repeatedly halving the search range. It runs in O(log n) time. The array must be sorted before you start. Off by one errors in the loop bounds are the most common mistake.

[Source: stack_queue]
A stack follows last in first out order, and push and pop take O(1) time. A queue follows first in first out order, and enqueue and dequeue take O(1) time. Stacks are used for undo features and tracking recursive calls. Queues are used in breadth first search and task scheduling.


In [6]:
MODEL_NAME = "gemini-3.5-flash-lite"

SYSTEM_PROMPT = (
    "You are a study assistant for data structures and algorithms. "
    "When you need facts, use the search_documents tool. "
    "Only answer based on what you find in the documents. "
    "If the documents do not contain the answer, say so clearly."
)
print("Model:", MODEL_NAME)

Model: gemini-3.5-flash-lite


In [7]:
import time
from google import genai
from google.genai import types

client = genai.Client()

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find the most relevant paragraphs.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=SYSTEM_PROMPT,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME, contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"

_ = run_agent("How fast is binary search and what must be true about the array?")

Question: How fast is binary search and what must be true about the array?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'binary search time complexity array sorted'})
  Result preview: [Source: binary_search]
Binary search finds a target in a sorted array by repeatedly halving the sea...

Final answer:
Based on the provided documents:

* **Speed (Time Complexity):** Binary search runs in **$O(\log n)$** time by repeatedly halving the search range.
* **Array Condition:** The array **must be sorted** before you start.


In [8]:
questions = [
    "Why is inserting in the middle of an array slow?",
    "Which data structure is used in breadth first search?",
    "What is the difference between a stack and a queue?",
    # Not answered by the documents:
    "How many ghosts eat carrots everyday?",
    # General knowledge:
    "What is the most popular food item in Canada?",
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stays under the free-tier rate limit
    print("\n" + "=" * 60)
    run_agent(q)


Question: Why is inserting in the middle of an array slow?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'inserting in the middle of an array slow'})
  Result preview: [Source: arrays]
Arrays store elements in contiguous memory. Accessing an element by index takes O(1...

Final answer:
Inserting in the middle of an array is slow—taking $O(n)$ time—because arrays store elements in contiguous memory. To make room for a new element at a specific index, all subsequent elements must be shifted over by one position.

Question: Which data structure is used in breadth first search?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'breadth first search data structure'})
  Result preview: [Source: stack_queue]
A stack follows last in first out order, and push and pop take O(1) time. A qu...

Final answer:
Based on the provided documents, a **queue** is used in breadth-first search.

Question: Wh

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)